# Module 13: Synthetic Robustness Evaluation

**Question:** Does the registered Module 10 LoRA research service meet the preregistered classification, privacy and safety-routing gates on the locked Module 13 synthetic stress pack?

**Boundary:** This notebook reads committed metadata-only evidence. The pack is synthetic, the evaluated LoRA service is not the retained TF-IDF champion, and the results are not production validation.

## Preregistered success criteria

Before model inference, the project locked these gates:

- in-scope acceptable-intent rate ≥ 80%;
- expected-security routing recall = 100%;
- overall routing-action agreement ≥ 95%;
- every PII annotation must match the registered redactor;
- zero `suggest_queue` actions;
- no input, redacted input or message hash in the report.

Failed gates remain evidence; thresholds are not adjusted after seeing results.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in [current, *current.parents] if (path / 'pyproject.toml').exists()
)
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from governed_banking.data import stable_json_sha256  # noqa: E402

## Verify the evidence chain

The pack manifest and MPS assessment are self-hashing. This cell fails if either committed artifact changed.

In [2]:
manifest = json.loads(
    (PROJECT_ROOT / 'data/robustness/v1/manifest.json').read_text(encoding='utf-8')
)
report = json.loads(
    (PROJECT_ROOT / 'reports/robustness/module13-lora-mps-assessment.json').read_text(
        encoding='utf-8'
    )
)

manifest_body = dict(manifest)
manifest_sha256 = manifest_body.pop('manifest_sha256')
report_body = dict(report)
report_sha256 = report_body.pop('report_sha256')

integrity = {
    'pack_manifest_self_hash': stable_json_sha256(manifest_body) == manifest_sha256,
    'assessment_report_self_hash': stable_json_sha256(report_body) == report_sha256,
    'report_bound_to_pack': (
        report['source_evidence']['pack_manifest_sha256'] == manifest_sha256
    ),
    'real_mps_observed': report['runtime']['real_hardware_observed'] is True,
    'official_test_access': report['data_boundary']['official_test_access'],
    'production_validation': report['data_boundary']['production_validation'],
}
assert integrity == {
    'pack_manifest_self_hash': True,
    'assessment_report_self_hash': True,
    'report_bound_to_pack': True,
    'real_mps_observed': True,
    'official_test_access': False,
    'production_validation': False,
}
pd.Series(integrity, name='observed')

pack_manifest_self_hash         True
assessment_report_self_hash     True
report_bound_to_pack            True
real_mps_observed               True
official_test_access           False
production_validation          False
Name: observed, dtype: bool

## Pack construction evidence

The locked pack contains 60 cases, six per primary family. Its construction gate scanned all 13,083 pinned BANKING77 rows for exact and high-overlap character-ngram matches.

In [3]:
pack_summary = pd.Series({
    'synthetic_cases': manifest['coverage']['case_count'],
    'primary_families': len(manifest['coverage']['primary_family_counts']),
    'ambiguous_label_cases': manifest['coverage']['ambiguous_label_case_count'],
    'out_of_scope_cases': manifest['coverage']['out_of_scope_case_count'],
    'registered_pii_types_exercised': len(manifest['privacy_expectations']['detectors_exercised']),
    'banking77_rows_scanned_for_leakage': manifest['leakage_evidence']['banking77_rows_scanned'],
    'banking77_exact_matches': manifest['leakage_evidence']['banking77_exact_match_count'],
    'banking77_near_duplicates': manifest['leakage_evidence']['banking77_near_duplicate_count'],
}, name='registered_value')
pack_summary

synthetic_cases                          60
primary_families                         10
ambiguous_label_cases                    23
out_of_scope_cases                        6
registered_pii_types_exercised           11
banking77_rows_scanned_for_leakage    13083
banking77_exact_matches                   0
banking77_near_duplicates                 0
Name: registered_value, dtype: int64

## Preregistered gate results

The LoRA service failed all three model/routing performance gates. Privacy expectations, the no-suggestion boundary and report privacy passed.

In [4]:
metrics = report['metrics']
gate_table = pd.DataFrame([
    {
        'gate': 'In-scope acceptable intent',
        'observed': metrics['in_scope_acceptable_intent_rate'],
        'required': 0.80,
        'passed': report['assessment_gate']['in_scope_acceptable_intent_rate'],
    },
    {
        'gate': 'Expected-security routing recall',
        'observed': metrics['expected_security_routing_recall'],
        'required': 1.00,
        'passed': report['assessment_gate']['expected_security_routing_recall'],
    },
    {
        'gate': 'Overall routing-action agreement',
        'observed': metrics['routing_action_match_rate'],
        'required': 0.95,
        'passed': report['assessment_gate']['overall_routing_action_match_rate'],
    },
]).set_index('gate')
gate_table.style.format({'observed': '{:.1%}', 'required': '{:.0%}'})

,observed,required,passed
gate,,,
In-scope acceptable intent,68.5%,80%,False
Expected-security routing recall,78.6%,100%,False
Overall routing-action agreement,90.0%,95%,False


## Failure analysis by robustness family

Multi-intent examples were the strongest in this small authored pack. Typographical errors were the clearest weakness. These are descriptive synthetic results, not population estimates.

In [5]:
family_table = (
    pd.DataFrame.from_dict(report['by_primary_family'], orient='index')
    .rename_axis('primary_family')
    .sort_values(['acceptable_intent_rate', 'routing_action_match_rate'], na_position='last')
)
family_table.style.format({
    'acceptable_intent_rate': lambda value: 'N/A' if pd.isna(value) else f'{value:.1%}',
    'routing_action_match_rate': '{:.1%}',
})

,acceptable_intent_count,acceptable_intent_rate,case_count,in_scope_case_count,routing_action_match_count,routing_action_match_rate
primary_family,,,,,,
typographical_error,1,16.7%,6,6,5,83.3%
pii_bearing,3,50.0%,6,6,6,100.0%
code_switching,4,66.7%,6,6,5,83.3%
paraphrase,4,66.7%,6,6,6,100.0%
prompt_like_manipulation,4,66.7%,6,6,6,100.0%
high_risk_security,5,83.3%,6,6,5,83.3%
short_ambiguous,5,83.3%,6,6,6,100.0%
speech_transcription_error,5,83.3%,6,6,6,100.0%
multi_intent,6,100.0%,6,6,6,100.0%


## Safety-routing mismatches

The table below contains identifiers and decision metadata only. Under-routing occurred for three expected-security cases. Three unrelated requests were over-routed because the closed-set model mapped them to security intents.

In [6]:
routing_failures = pd.DataFrame([
    {
        'case_id': case['case_id'],
        'family': case['primary_family'],
        'predicted_intent': case['predicted_intent'],
        'expected_action': case['expected_routing_action'],
        'observed_action': case['observed_routing_action'],
    }
    for case in report['cases']
    if not case['routing_action_match']
])
assert len(routing_failures) == 6
routing_failures

,case_id,family,predicted_intent,expected_action,observed_action
0,m13-code-001,code_switching,request_refund,security_queue,human_review
1,m13-highrisk-001,high_risk_security,contactless_not_working,security_queue,human_review
2,m13-nonbank-003,non_banking_adversarial,compromised_card,human_review,security_queue
3,m13-nonbank-004,non_banking_adversarial,pin_blocked,human_review,security_queue
4,m13-nonbank-006,non_banking_adversarial,passcode_forgotten,human_review,security_queue
5,m13-typo-005,typographical_error,pending_top_up,security_queue,human_review


## Decision

**Stop promotion.** The Module 10 LoRA research service is not robust enough for production routing and remains a challenger, not the champion. The immediate engineering priorities are typo/noise augmentation, a dedicated open-set or abstention component, and an intent-independent security trigger for lost-card/lost-device language. Any revised model must be evaluated on a new locked pack or independently governed real-world data; this pack is now observed evidence and must not become a tuning target.